# CV07: 64-D Triplet + KNN

Only the **Triplet + KNN** comparison from CV07 is included. Thesis sequence: focal-trained WideResNet → frozen 2048-D features → separate fully connected 64-D projection → triplet loss → training-only KNN.

The rendered thesis equation (PDF p. 77) is `mean(max(0, squared_distance(a,p) - squared_distance(a,n) + margin))`. The CV method is on PDF pp. 94-95; Table 4.7 lists Triplet + KNN on p. 101. The original margin, mining, normalization, optimizer, epoch budget and KNN settings are unavailable; defaults here are recorded project choices. The thesis's 62 ± 1.9% is not a result of this notebook.

Run the focal baseline notebook first, using the same dataset and manifest. Expert Control/Parkinson/Unknown labels define triplets. Clinical labels are used for reports only. Unknown is a third class; vote fractions are not calibrated probabilities.

**Colab preparation:** copy `outputs/colab_upload/parkinson_training_code_v4_cv07.zip` from this project into `My Drive/ParkinsonResearch/`. Setup checks its checksum. The shared dataset and focal runs use their existing Drive paths. The notebook also works locally from the repository.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True


def extract_checked(source, destination, expected_sha256):
    if not source.is_file():
        raise FileNotFoundError(f'Missing Drive file: {source}. Check the Google account and whether the code archive upload is complete.')
    local_zip = Path('/content') / source.name
    shutil.copy2(source, local_zip)
    if hashlib.sha256(local_zip.read_bytes()).hexdigest() != expected_sha256:
        raise ValueError(f'Archive checksum differs from this notebook: {source.name}')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError(f'Unsafe archive path: {name}')
        archive.extractall(destination)


DRIVE_DIR = None
if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/ParkinsonResearch')
    DATA_ROOT = Path('/content/ParkinsonData')
    extract_checked(
        DRIVE_DIR / 'datasets/parkinson_dataset_v1.zip', DATA_ROOT,
        '85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6',
    )
    metadata = json.loads((DATA_ROOT / 'dataset_package.json').read_text())
    for name, info in metadata['files'].items():
        target = (DATA_ROOT / name).resolve()
        if not target.is_relative_to(DATA_ROOT.resolve()):
            raise ValueError(f'Unsafe dataset path: {name}')
        if hashlib.sha256(target.read_bytes()).hexdigest() != info['sha256']:
            raise ValueError(f'Dataset verification failed: {name}')
    PROJECT_DIR = Path('/content/ParkinsonAiResearch')
    extract_checked(DRIVE_DIR / 'parkinson_training_code_v4_cv07.zip', PROJECT_DIR,
                    'f4d38bae464e8828aa07c4d1b5ad738dc546de4fad900bd9fe158f4d1ec0845a')
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_DIR = next((p for p in candidates if (p / 'train.py').is_file()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Open this notebook from the ParkinsonAiResearch repository.')
    DATA_ROOT = PROJECT_DIR

DATASET_DIR = DATA_ROOT / 'datasets'
MANIFEST = DATASET_DIR / 'annotations/fold_manifest.csv'
if not MANIFEST.is_file():
    raise FileNotFoundError(f'Dataset manifest not found: {MANIFEST}')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError('Select Kernel → Colab → New Colab Server, choose a GPU, then rerun setup.')
print('Code directory:', PROJECT_DIR)
print('Dataset directory:', DATASET_DIR)
print('Fold manifest:', MANIFEST)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'MPS — Apple GPU' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
      else 'CPU')
print('Ready. Configure CV07, then run training and development selection.')


Start with fold 1. After the pilot succeeds, set `FOLD_END = 25`. Each fold uses its matching source focal checkpoint and an independent projection. Saves after every projection update allow resume; feature extraction repeats after a runtime reset. Keep source baseline files available.

Defaults: linear 2048→64 projection, no normalization, squared-L2 triplet margin 1, all valid ordered triplets including inactive terms, AdamW lr 0.001, weight decay 0, one full-fold update per epoch. `batch_hard` and cross-participant positives are optional project ablations. These settings are not recovered thesis hyperparameters. Change `RECIPE` when changing settings.


In [ ]:
FOLD_START = 1
FOLD_END = 1  # change to 25 after the pilot
EPOCHS = 30
SEED = 42
MARGINS = [1.0]  # optional development grid: [0.5, 1.0, 2.0]
TRIPLET_MINING = 'all'  # optional ablation: 'batch_hard'
POSITIVE_POLICY = 'all'  # optional ablation: 'cross_participant'
NORMALIZE = False
K_VALUES = [1, 3, 5, 7]
DISTANCES = ['l2']
WEIGHTINGS = ['uniform', 'distance']
DEVICE = 'cuda' if torch.cuda.is_available() else 'auto'
BASELINE_DIR = (DRIVE_DIR / 'runs/baselines_shared_dataset_v1/focal/checkpoints'
                if IN_COLAB else PROJECT_DIR / 'outputs/baselines/focal/checkpoints')
BASELINE_GLOB = '*_last.pt'
RECIPE = 'triplet_all_raw_v1'
OUTPUT_ROOT = (DRIVE_DIR / 'runs/cv07' / RECIPE if IN_COLAB
               else PROJECT_DIR / 'outputs/cv07' / RECIPE)
SELECTION_DIR = OUTPUT_ROOT.parent / f'{RECIPE}_selection_folds{FOLD_START:02d}-{FOLD_END:02d}'
if not BASELINE_DIR.is_dir():
    raise FileNotFoundError(f'Finish the focal baselines or correct BASELINE_DIR: {BASELINE_DIR}')
print('Source focal checkpoints:', BASELINE_DIR)
print('CV07 training output:', OUTPUT_ROOT)
print('Development selection output:', SELECTION_DIR)


Train one projection per source fold and margin candidate. The runner validates all source matches before starting, logs class/triplet support, and resumes compatible saved runs. Each KNN bank contains training drawings only. Training and selection open no test images.


In [ ]:
command = [sys.executable, '-u', 'train_triplet_knn_all_folds.py',
           '--baseline-dir', str(BASELINE_DIR), '--baseline-glob', BASELINE_GLOB,
           '--manifest', str(MANIFEST), '--root', str(DATA_ROOT),
           '--output-root', str(OUTPUT_ROOT), '--fold-start', str(FOLD_START),
           '--fold-end', str(FOLD_END), '--epochs', str(EPOCHS), '--seed', str(SEED),
           '--embedding-dim', '64', '--triplet-mining', TRIPLET_MINING,
           '--positive-policy', POSITIVE_POLICY,
           '--normalize' if NORMALIZE else '--no-normalize',
           '--device', DEVICE, '--resume', '--margins', *map(str, MARGINS)]
subprocess.run(command, check=True)


Select one shared margin/projection/KNN configuration by pooled three-class expert macro-F1. Every candidate needs every requested final fold checkpoint. The selector verifies OOF drawing coverage and writes the candidate table, image/participant reports and frozen `foldXX/selected.pt` checkpoints. Oversized k values are skipped and recorded. Fold 22 retains an empty validation report.

Scores are **development selection scores**, optimistic after tuning; pilot scores are incomplete. Use a new selection directory for a new selection run.


In [ ]:
command = [sys.executable, '-u', 'tune_triplet_knn.py',
           '--runs-dir', str(OUTPUT_ROOT), '--manifest', str(MANIFEST),
           '--root', str(DATA_ROOT), '--output-dir', str(SELECTION_DIR),
           '--fold-start', str(FOLD_START), '--fold-end', str(FOLD_END),
           '--device', DEVICE, '--k-values', *map(str, K_VALUES),
           '--distances', *DISTANCES, '--weightings', *WEIGHTINGS]
subprocess.run(command, check=True)
selection = json.loads((SELECTION_DIR / 'selection.json').read_text())
report = json.loads((SELECTION_DIR / 'oof_cv07_report.json').read_text())
print('Selected configuration:', selection['selected'])
print('Complete development schedule:', selection['complete_development_schedule'])
print('Expert image metrics:', report['image_expert_metrics'])
print('Clinical participant metrics:', report['participant_clinical_metrics'])


Independently reload the first selected model and reproduce its saved validation predictions. This does not evaluate the held-out test partition. A future frozen test protocol can use the evaluator's explicit test flag. A thesis-style weighted 25-model ensemble requires a separately specified weighting protocol.


In [ ]:
checkpoint = SELECTION_DIR / f'fold{FOLD_START:02d}' / 'selected.pt'
reload_dir = SELECTION_DIR / f'reloaded_fold{FOLD_START:02d}'
subprocess.run([sys.executable, '-u', 'eval_representation.py',
                '--checkpoint', str(checkpoint), '--manifest', str(MANIFEST),
                '--root', str(DATA_ROOT), '--device', DEVICE,
                '--output-dir', str(reload_dir)], check=True)
original = checkpoint.parent / 'val_predictions.csv'
assert original.read_bytes() == (reload_dir / 'val_predictions.csv').read_bytes()
print('Independent reload reproduced the saved validation predictions.')


Optional new-image prediction takes no expert or clinical query labels. Supply PNG/JPEG file paths below, then run the cell. The source baseline and matching manifest must remain available. Predictions include KNN vote fractions and source hashes.


In [ ]:
NEW_IMAGES = []  # example: [Path('/content/new_spiral.png')]
if NEW_IMAGES:
    prediction_dir = SELECTION_DIR / 'new_images'
    subprocess.run([sys.executable, '-u', 'predict_triplet_knn.py',
                    '--checkpoint', str(checkpoint), '--manifest', str(MANIFEST),
                    '--root', str(DATA_ROOT), '--device', DEVICE,
                    '--images', *map(str, NEW_IMAGES), '--output-dir', str(prediction_dir)], check=True)
    print((prediction_dir / 'predictions.csv').read_text(encoding='utf-8-sig'))
else:
    print('Set NEW_IMAGES to run optional unlabeled image inference.')


See `README/CV07_TRIPLET_KNN.md` for the objective, defaults, provenance, commands and verification limits. Software tests use synthetic inputs; research-cohort training and clinical accuracy are not established by those tests.
